# 01_surrogate_baseline: Step 1 - surrogate baselines: ablation of the data steps, default model zoo,

```
and a replication of the Lyu et al. (2025) protocol.

A. Ablation (held-out test days, ExtraTrees & XGBoost defaults):
   raw 5-min -> + outlier smoothing -> + hourly aggregation -> + time-of-day/schedule
B. Default model zoo + weighted ensemble on the deployed (hourly) data
C. Paper protocol: office-1, first 12 weekdays, 5-min, smoothing, 7 features,
   random 70/15/15 row split (as AutoGluon-Tabular does by default)
Outputs -> outputs/surrogate/
```

Dikonversi dari `scripts/01_surrogate_baseline.py`. Jalankan dari folder `notebooks/`.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve().parent   # notebook runs from AI-OBEP-MLPaper/notebooks
sys.path.insert(0, str(ROOT / "src"))
from obep_mlpaper.data import (FEATURES, FEATURES_PAPER, TARGET, ROOM_AREA, add_time_features,
                               day_split, five_minute_dataset, hourly_dataset, load_room)
from obep_mlpaper.surrogate import WeightedEnsemble, metrics, model_zoo

out = ROOT / "outputs" / "surrogate"
out.mkdir(parents=True, exist_ok=True)

In [2]:
def fit_eval(name, make, Xtr, ytr, Xte, yte, op=None):
    m = make().fit(Xtr, ytr)
    p = np.clip(m.predict(Xte), 0, None)
    row = dict(model=name, **metrics(yte, p))
    if op is not None:
        row["R2_operating"] = metrics(yte[op], p[op])["R2"]
    return row, m, p

In [3]:
# ---------------------------------------------------------------- A. ablation
abl = []
steps = []
raw = add_time_features(load_room())
raw["y"] = raw["energy_kwh"] * 12 * 1000 / ROOM_AREA
steps.append(("1 raw 5-min, 7 paper features", raw, FEATURES_PAPER))
sm = five_minute_dataset(smooth=True)
sm["y"] = sm["target_kwh"] * 12 * 1000 / ROOM_AREA
steps.append(("2 + outlier smoothing (Hampel, adjacent mean)", sm, FEATURES_PAPER))
hr = hourly_dataset()
hr["y"] = hr[TARGET]
steps.append(("3 + hourly aggregation", hr, FEATURES_PAPER))
steps.append(("4 + time of day & HVAC schedule (deployed)", hr, FEATURES))
zoo = model_zoo()
for label, d, feats in steps:
    tr, te = day_split(d)
    X, y = d[feats].to_numpy(), d["y"].to_numpy()
    op = d["hvac_schedule"].to_numpy()[te] == 1
    for name in ("ExtraTrees", "XGBoost"):
        row, _, _ = fit_eval(name, lambda: model_zoo()[name], X[tr], y[tr], X[te], y[te], op)
        abl.append(dict(step=label, **row))
        print(f"{label:48s} {name:10s} R2={row['R2']:.3f}  R2_op={row['R2_operating']:.3f}  "
              f"CV(RMSE)={row['CVRMSE_pct']:.1f}%")
pd.DataFrame(abl).to_csv(out / "ablation_data_steps.csv", index=False)

1 raw 5-min, 7 paper features                    ExtraTrees R2=0.623  R2_op=0.092  CV(RMSE)=68.4%


1 raw 5-min, 7 paper features                    XGBoost    R2=0.575  R2_op=-0.037  CV(RMSE)=72.6%


2 + outlier smoothing (Hampel, adjacent mean)    ExtraTrees R2=0.626  R2_op=0.088  CV(RMSE)=68.0%
2 + outlier smoothing (Hampel, adjacent mean)    XGBoost    R2=0.591  R2_op=-0.031  CV(RMSE)=71.1%


3 + hourly aggregation                           ExtraTrees R2=0.672  R2_op=0.169  CV(RMSE)=62.6%
3 + hourly aggregation                           XGBoost    R2=0.577  R2_op=-0.034  CV(RMSE)=71.1%


4 + time of day & HVAC schedule (deployed)       ExtraTrees R2=0.675  R2_op=0.148  CV(RMSE)=62.3%
4 + time of day & HVAC schedule (deployed)       XGBoost    R2=0.615  R2_op=-0.011  CV(RMSE)=67.8%


In [4]:
# ---------------------------------------------------------------- B. zoo on deployed data
d = hourly_dataset()
tr, te = day_split(d)
X, y, g = d[FEATURES].to_numpy(), d[TARGET].to_numpy(), d["day"].to_numpy()
op = d["hvac_schedule"].to_numpy()[te] == 1
rows = []
for name in zoo:
    row, _, _ = fit_eval(name, lambda: model_zoo()[name], X[tr], y[tr], X[te], y[te], op)
    rows.append(row)
makers = {k: (lambda k=k: model_zoo()[k]) for k in ("ExtraTrees", "RandomForest", "LightGBM", "XGBoost", "KNN", "MLP")}
ens = WeightedEnsemble(makers).fit(X[tr], y[tr], g[tr])
p = np.clip(ens.predict(X[te]), 0, None)
rows.append(dict(model="WeightedEnsemble", **metrics(y[te], p), R2_operating=metrics(y[te][op], p[op])["R2"]))
zoo_res = pd.DataFrame(rows)
zoo_res.to_csv(out / "baseline_zoo_test_metrics.csv", index=False)
print("\nDefault zoo (hourly, held-out days), ensemble weights:", ens.weight_dict())
print(zoo_res.round(3).to_string(index=False))


Default zoo (hourly, held-out days), ensemble weights: {'ExtraTrees': 0.0, 'RandomForest': 0.618, 'LightGBM': 0.0, 'XGBoost': 0.274, 'KNN': 0.108, 'MLP': 0.0}
           model    R2  RMSE   MAE  MAPE_pct  CVRMSE_pct  NMBE_pct  R2_operating
           Ridge 0.597 8.480 6.104    70.802      69.371   -24.426        -0.020
             KNN 0.653 7.871 4.582    63.760      64.392   -25.400         0.093
    RandomForest 0.675 7.620 4.655    62.919      62.337   -27.196         0.147
      ExtraTrees 0.675 7.619 4.540    62.596      62.329   -26.505         0.148
        LightGBM 0.662 7.771 4.793    60.986      63.568   -26.990         0.121
         XGBoost 0.615 8.285 4.843    66.745      67.779   -29.224        -0.011
             MLP 0.640 8.015 5.223    62.560      65.569   -23.411         0.081
WeightedEnsemble 0.666 7.722 4.663    63.897      63.173   -27.557         0.123


In [5]:
# ---------------------------------------------------------------- C. paper protocol
p12 = five_minute_dataset(n_days=12, smooth=True)
rng = np.random.RandomState(0)
idx = rng.permutation(len(p12))
n = len(idx)
itr, iva, ite = idx[:int(.7 * n)], idx[int(.7 * n):int(.85 * n)], idx[int(.85 * n):]
Xp, yp = p12[FEATURES_PAPER].to_numpy(), p12["target_kwh"].to_numpy()
prow, models = [], {}
for name in zoo:
    row, m, _ = fit_eval(name, lambda: model_zoo()[name], Xp[itr], yp[itr], Xp[ite], yp[ite])
    prow.append(row)
    models[name] = m
# AutoGluon-style L2 weighted ensemble: weights fit on the validation rows
from scipy.optimize import nnls
members = [k for k in models if k != "Ridge"]
Pv = np.column_stack([models[k].predict(Xp[iva]) for k in members])
w, _ = nnls(Pv, yp[iva])
w = w / w.sum()
pt = np.clip(np.column_stack([models[k].predict(Xp[ite]) for k in members]) @ w, 0, None)
prow.append(dict(model="WeightedEnsemble_L2", **metrics(yp[ite], pt)))
prow.append(dict(model="Lyu et al. 2025 (AutoGluon WeightedEnsemble_L2)", R2=0.933, MAPE_pct=26.094))
prow.append(dict(model="Lyu et al. 2025 (LSTM)", R2=0.921, MAPE_pct=19.358))
paper = pd.DataFrame(prow)
paper.to_csv(out / "paper_protocol_metrics.csv", index=False)
print("\nPaper protocol (12 days, 5-min, random rows):")
print(paper[["model", "R2", "RMSE", "MAPE_pct", "CVRMSE_pct"]].round(3).to_string(index=False))


Paper protocol (12 days, 5-min, random rows):
                                          model    R2  RMSE  MAPE_pct  CVRMSE_pct
                                          Ridge 0.711 0.083    28.426      53.803
                                            KNN 0.942 0.037    12.721      24.152
                                   RandomForest 0.958 0.032    11.248      20.428
                                     ExtraTrees 0.969 0.027     9.156      17.696
                                       LightGBM 0.963 0.030    11.411      19.137
                                        XGBoost 0.957 0.032    11.740      20.804
                                            MLP 0.951 0.034    11.015      22.107
                            WeightedEnsemble_L2 0.967 0.028     9.684      18.276
Lyu et al. 2025 (AutoGluon WeightedEnsemble_L2) 0.933   NaN    26.094         NaN
                         Lyu et al. 2025 (LSTM) 0.921   NaN    19.358         NaN
